In [1]:
import numpy as np
import cv2
import os
import glob
import warnings
from tqdm import tqdm
from skimage.metrics import structural_similarity as ssim
from skimage.segmentation import slic

warnings.filterwarnings("ignore")

# =============================================================================
# 1. HAMACHER-BASED IFS (Illumination Recovery)
# =============================================================================
class IFSHamacher_v19:
    def __init__(self, beta=0.42): # Slightly adjusted beta for smoother base
        self.beta = beta

    def apply(self, layer):
        x = layer.astype(np.float32) / 255.0
        mu = x / (self.beta + (1 - self.beta) * x + 1e-6)
        nu = (1 - x) / (1 + (1 - self.beta) * x + 1e-6)
        pi = np.clip(1 - mu - nu, 0, 1)
        # Smooth boost for the base illumination layer
        enh = mu + pi * (x**0.48)
        return (np.clip(enh, 0, 1) * 255).astype(np.uint8)

# =============================================================================
# 2. TEXTURE-ADAPTIVE SPECTRAL SHARPENING
# =============================================================================
def adaptive_spectral_sharpen(detail_layer, n_segments=650):
    d_min, d_max = detail_layer.min(), detail_layer.max()
    d_norm = (detail_layer - d_min) / (d_max - d_min + 1e-6)
    
    # Calculate local energy (variance) to gate sharpening
    energy = cv2.GaussianBlur(d_norm**2, (7, 7), 0)
    energy = cv2.normalize(energy, None, 0, 1, cv2.NORM_MINMAX)

    segments = slic(d_norm, n_segments=n_segments, compactness=12, start_label=0, channel_axis=None)
    num_nodes = segments.max() + 1
    node_means = np.array([d_norm[segments == i].mean() for i in range(num_nodes)])

    # Symmetric Normalized Adjacency Matrix
    W = np.exp(-(node_means[:, None] - node_means[None, :])**2 / 0.08)
    d_inv_sqrt = np.power(np.sum(W, axis=1) + 1e-6, -0.5)
    S = (W * d_inv_sqrt).T * d_inv_sqrt
    
    refined_nodes = S.dot(node_means)
    refined_detail = refined_nodes[segments]
    
    # Adaptive mixing: Sharpen more where energy is high (real texture)
    # and less where energy is low (potential noise)
    final_detail = d_norm + (refined_detail - d_norm) * energy
    return (final_detail * (d_max - d_min) + d_min).astype(np.float32)

# =============================================================================
# V19 PIPELINE
# =============================================================================
def fire_v19_adaptive_pyramid(img):
    if img is None: return None

    # Step 1: High-fidelity Denoising
    img_dn = cv2.fastNlMeansDenoisingColored(img, None, 3, 3, 7, 21)
    
    # Step 2: Frequency Decomposition
    # Base: Global Illumination | Detail: Local Texture
    base = cv2.GaussianBlur(img_dn, (13, 13), 0)
    
    # Step 3: Enhance Base via IFS
    base_enh = IFSHamacher_v19().apply(base)
    
    # Step 4: Adaptive Spectral Sharpening on Detail
    # Extract V-channel detail for structural sharpening
    hsv_dn = cv2.cvtColor(img_dn, cv2.COLOR_BGR2HSV).astype(np.float32)
    v_base = cv2.GaussianBlur(hsv_dn[:, :, 2], (13, 13), 0)
    v_detail = hsv_dn[:, :, 2] - v_base
    v_detail_refined = adaptive_spectral_sharpen(v_detail)
    
    # Step 5: Reconstruction
    hsv_base = cv2.cvtColor(base_enh, cv2.COLOR_BGR2HSV)
    v_combined = hsv_base[:, :, 2].astype(np.float32) + 0.85 * v_detail_refined
    
    # Precision Guided Filter to lock SSIM
    guide = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32)
    v_final = cv2.ximgproc.guidedFilter(guide, v_combined, radius=2, eps=1e-4)
    
    hsv_base[:, :, 2] = np.clip(v_final, 0, 255).astype(np.uint8)
    enhanced = cv2.cvtColor(hsv_base, cv2.COLOR_HSV2BGR)

    # Step 6: Lab Color Calibration
    lab_orig = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype(np.float32)
    lab_enh = cv2.cvtColor(enhanced, cv2.COLOR_BGR2LAB).astype(np.float32)
    l_o, a_o, b_o = cv2.split(lab_orig)
    l_e, a_e, b_e = cv2.split(lab_enh)
    for enh_ch, orig_ch in [(a_e, a_o), (b_e, b_o)]:
        m_o, s_o = cv2.meanStdDev(orig_ch)
        m_e, s_e = cv2.meanStdDev(enh_ch)
        enh_ch[:] = (enh_ch - m_e.flatten()) * (s_o.flatten() / (s_e.flatten() + 1e-6)) + m_o.flatten()

    res = cv2.cvtColor(np.clip(cv2.merge([l_e, a_e, b_e]), 0, 255).astype(np.uint8), cv2.COLOR_LAB2BGR)

    # Step 7: Final Dynamic Range Anchor
    res_gray = cv2.cvtColor(res, cv2.COLOR_BGR2GRAY)
    target_mean = 115.5 # Fine-tuned anchor
    res = np.clip(res * (target_mean / (np.mean(res_gray) + 1e-6)), 0, 255).astype(np.uint8)

    return cv2.addWeighted(res, 0.97, img, 0.03, 0)

# =============================================================================
# BENCHMARK
# =============================================================================
def run_benchmark(configs):
    for cfg in configs:
        print(f"\n>>> Running V19 Adaptive Pyramid: {cfg['name']}")
        lows = sorted(glob.glob(os.path.join(cfg['low'], "*.png")))
        highs = sorted(glob.glob(os.path.join(cfg['high'], "*.png")))
        psnrs, ssims = [], []
        for lp, hp in tqdm(zip(lows, highs), total=len(lows)):
            low, high = cv2.imread(lp), cv2.imread(hp)
            if low is None or high is None: continue
            high = cv2.resize(high, (low.shape[1], low.shape[0]))
            enhanced = fire_v19_adaptive_pyramid(low)
            eg, hg = cv2.cvtColor(enhanced, cv2.COLOR_BGR2GRAY), cv2.cvtColor(high, cv2.COLOR_BGR2GRAY)
            mse = np.mean((eg.astype(np.float64) - hg.astype(np.float64)) ** 2)
            psnrs.append(20 * np.log10(255.0 / (np.sqrt(mse) + 1e-6)))
            ssims.append(ssim(eg, hg, data_range=255))
        print(f"[{cfg['name']}] Avg PSNR: {np.mean(psnrs):.2f} | Avg SSIM: {np.mean(ssims):.4f}")

if __name__ == "__main__":
    datasets = [{
        "name": "LOLv1_Adaptive_Pyramid",
        "low": r"C:\Users\divya\Downloads\LOLdataset\LOLdataset\eval15\low",
        "high": r"C:\Users\divya\Downloads\LOLdataset\LOLdataset\eval15\high"
    }]
    run_benchmark(datasets)


>>> Running V19 Adaptive Pyramid: LOLv1_Adaptive_Pyramid


100%|██████████████████████████████████████████████████████████████████████████████████| 15/15 [00:28<00:00,  1.90s/it]

[LOLv1_Adaptive_Pyramid] Avg PSNR: 16.56 | Avg SSIM: 0.7185
